# Tag 7

- VIEW
  - gespeicherte Abfragestruktur
  - hiermit kann eine komplexe Abfrage aufgerufen werden, als wäre es eine einzelne Tabelle
  - Benutzer/Anwendungen mit Leserecht auf das View brauchen keine Rechte auf die Tabellen
  - Das View selbst darf in seiner äußeren Abfrage keine ORDER BY Klausel haben.
  - Die Anweisung CREATE OR ALTER VIEW steht allein in eine Batch (extra Zelle bzw. GO davor und danach)
- FUNCTION
  - gibt einen Wert zurück (Skalarwertfunktion) oder mehrere Werte (Tabellenwertfunktion)
  - Skalarwertfunktion wird in der SELECT Klausel aufgerufen
  - Tabellenwertfunktion wird in der FROM Klausel aufgerufen
  - bekommt Parameter übergeben, z. Bsp. Filterkriterien
- TRIGGER
  - Programm, das auf Ereignisse reagiert
  - BEFORE, AFTER, INSTEAD OF
  - Keine UPDATE Trigger möglich bei Temporal Tables
- PROCEDURE
  - Ein Programm, dass mit EXEC aufgerufen wird
  - bekommt Parameter übergeben
  - führt Anweisungen aus
  - hat keinen Rückgabewert

## Ideen für den UPDATE Trigger
 
- Wird ausgelöst INSTEAD OF UPDATE
- Es darf immer nur eine Zeile geändert werden, nicht mehrere
- Abbruch, wenn die Zeile nicht gefunden wurde (identifiziert durch PersonalNr und ProjNr kombiniert)
- Es darf nur die Anzahl Stunden geändert werden, nicht die anderen Spalten!
- Die Stundenzahl darf nur erhöht werden.
- Das UPDATE wird nur dann durchgeführt, wenn noch ausreichen Budget vorhanden ist.
- Die Projekttabelle muss ebenfalls aktualisiert werden.
- Nutzen Sie die OUTPUT Klausel der UPDATE Anweisung, um die Änderungen anzuzeigen.

In [6]:
CREATE OR ALTER TRIGGER dbo.tr_123xyz
ON dbo.Arbeitet_An
INSTEAD OF UPDATE
AS
BEGIN
    SET NOCOUNT ON;
    BEGIN TRY
        BEGIN TRANSACTION;

        DECLARE @Kosten DECIMAL(9,2), @Restbudget DECIMAL(9,2), @Test INT;
        DECLARE @StundenDiff INT, @ProjNr INT;

        DECLARE @Posten AS TABLE(
          PersonalNr INT NOT NULL,
          ProjNr INT NOT NULL,
          TätigkeitsNr_alt INT NOT NULL,
          TätigkeitsNr_neu INT NOT NULL,
          Stunden_alt INT NOT NULL,
          Stunden_neu INT NOT NULL
        );

        INSERT INTO @Posten (PersonalNr, ProjNr, TätigkeitsNr_alt, TätigkeitsNr_neu, Stunden_alt, Stunden_neu)
        SELECT i.PersonalNr, i.ProjNr, d.TätigkeitsNr, i.TätigkeitsNr, d.Stunden, i.Stunden
        FROM inserted AS i
        JOIN deleted AS d on i.PersonalNr = d.PersonalNr AND i.ProjNr = d.ProjNr;

        SELECT @Test = COUNT(*) FROM inserted;
        IF @Test > 1 THROW 50000, 'Fehler: Es darf immer nur eine Zeile geändert werden', 1;

        SELECT @Test = COUNT(*) FROM @Posten;
        IF @Test = 0 THROW 50000, 'Fehler: Der Datensatz wurde nicht gefunden', 1;

        SELECT @Test = COUNT(*) FROM @Posten WHERE TätigkeitsNr_alt <> TätigkeitsNr_neu;
        IF @Test > 0 THROW 50000, 'Fehler: Es darf nur die Stundenzahl geändert werden', 1;

        SELECT @Test = COUNT(*) FROM @Posten WHERE Stunden_neu <= Stunden_alt;
        IF @Test > 0 THROW 50000, 'Fehler: Die Stundenzahl darf nur erhöht werden', 1;

        SELECT @StundenDiff = Stunden_neu - Stunden_alt, @ProjNr = ProjNr FROM @Posten;

        SELECT @Kosten = (t.Stundenlohn * @StundenDiff) FROM @Posten AS po JOIN Tätigkeit AS t ON po.TätigkeitsNr_neu = t.TätigkeitsNr;
        SELECT @Restbudget = (pr.Personalbudget - pr.Personalkostenstand) FROM Projekt AS pr WHERE pr.ProjNr = @ProjNr;
        IF @Kosten > @Restbudget THROW 50000, 'Fehler: Kein ausreichendes Budget verfügbar', 1;

        UPDATE Projekt
        SET Personalkostenstand += @Kosten
        WHERE ProjNr = @ProjNr;

        UPDATE aa
        SET aa.Stunden = po.Stunden_neu
        OUTPUT deleted.PersonalNr, deleted.ProjNr, deleted.Stunden AS [Stunden alt], inserted.Stunden AS [Stunden neu]
        FROM Arbeitet_An AS aa
        JOIN @Posten AS po ON aa.PersonalNr = po.PersonalNr AND aa.ProjNr = po.ProjNr;

        PRINT 'Yeah! Die Stunden und der Konstostand sind aktualisiert';

        COMMIT TRANSACTION;
    END TRY
    BEGIN CATCH
        ROLLBACK TRANSACTION;
        THROW;
    END CATCH;
    SET NOCOUNT OFF;
END;

Commands completed successfully.

Total execution time: 00:00:00.035

In [7]:
SELECT * FROM Arbeitet_An;
SELECT ProjNr, Personalbudget, Personalkostenstand FROM Projekt;

(9 rows affected)
(3 rows affected)

PersonalNr | ProjNr | TätigkeitsNr | Stunden
-----------+--------+--------------+--------
1          | 2      | 2            | 1000   
2          | 1      | 1            | 25     
2          | 3      | 2            | 80     
3          | 1      | 2            | 55     
3          | 3      | 2            | 65     
4          | 1      | 2            | 70     
4          | 2      | 1            | 25     
4          | 3      | 1            | 25     
5          | 2      | 3            | 160    
(9 rows)

ProjNr | Personalbudget | Personalkostenstand
-------+----------------+--------------------
1      | 10000.00       | 5000.00            
2      | 11000.00       | 6850.00            
3      | 10000.00       | 5600.00            
(3 rows)

Total execution time: 00:00:00.075

In [21]:
-- Was gibt's schon in Arbeitet_An?
SELECT * FROM Arbeitet_An;

(9 rows affected)

PersonalNr | ProjNr | TätigkeitsNr | Stunden
-----------+--------+--------------+--------
1          | 2      | 2            | 1000   
2          | 1      | 1            | 27     
2          | 3      | 2            | 80     
3          | 1      | 2            | 55     
3          | 3      | 2            | 65     
4          | 1      | 2            | 70     
4          | 2      | 1            | 25     
4          | 3      | 1            | 25     
5          | 2      | 3            | 160    
(9 rows)

Total execution time: 00:00:00.048

In [22]:
-- Würde mehrere Zeilen ändern
UPDATE Arbeitet_An
SET Stunden = 50;

Msg 50000, Level 16, State 1, Procedure tr_123xyz, Line 28 [Batch Start Line 1]
Fehler: Es darf immer nur eine Zeile geändert werden

Total execution time: 00:00:00.029

In [23]:
-- Würde die Art der Tätigkeit verändern
UPDATE Arbeitet_An
SET Stunden = 50 , TätigkeitsNr = 2
WHERE PersonalNr = 2 AND ProjNr = 1;

Msg 50000, Level 16, State 1, Procedure tr_123xyz, Line 34 [Batch Start Line 1]
Fehler: Es darf nur die Stundenzahl geändert werden

Total execution time: 00:00:00.006

In [24]:
-- Neue Stundenzahl ist kleiner als die alte
UPDATE Arbeitet_An
SET Stunden = 20
WHERE PersonalNr = 2 AND ProjNr = 1;

Msg 50000, Level 16, State 1, Procedure tr_123xyz, Line 37 [Batch Start Line 1]
Fehler: Die Stundenzahl darf nur erhöht werden

Total execution time: 00:00:00.005

In [25]:
-- Zu viele Stunden für das Budget
UPDATE Arbeitet_An
SET Stunden = 2000
WHERE PersonalNr = 2 AND ProjNr = 1;

Msg 50000, Level 16, State 1, Procedure tr_123xyz, Line 43 [Batch Start Line 1]
Fehler: Kein ausreichendes Budget verfügbar

Total execution time: 00:00:00.011

In [29]:
-- Dieses Update sollte erfolgreich ausgeführt werden
UPDATE Arbeitet_An
SET Stunden += 1
WHERE PersonalNr = 2 AND ProjNr = 1;

Yeah! Die Stunden und der Konstostand sind aktualisiert
(1 row affected)

PersonalNr | ProjNr | Stunden alt | Stunden neu
-----------+--------+-------------+------------
2          | 1      | 30          | 31         
(1 row)

Total execution time: 00:00:00.028